# PatchTST Reduced-Feature Model

Variants trained so far, all on the same reduced feature set: **`return_1h` + OHLC (5 channels)**, since `patch_tst_feature_importance.ipynb` found (two independent seeds) these were the only channels clearly, consistently helpful -- momentum/lagged-returns were actively harmful, volume/VIX were noise.

1. **Channel-0-only NLL baseline** (commit `eaaa35e`) -- same `channel_attention=True` / StudentT-NLL / mean-pooling setup as every other notebook in this project, loss restricted to `return_1h` only. Reproduced the 21-channel baseline almost exactly -- no improvement from dropping the harmful channels on retrain.
2. **Multi-channel NLL loss** -- averaged the same StudentT-NLL loss across all 5 channels instead of channel 0 only. **Negative result, reverted**: the loss curve did show real convergence instead of flat/noise, but only because OHLC is far easier to predict than `return_1h` -- the optimizer minimized total NLL mostly by nailing OHLC while `return_1h` degraded into a lag-echo/persistence strategy (`corr(loc_raw, y[t-1])` jumped to 0.42, the same pathology Run 1 in `docs/experiments.md` was diagnosed with and Run 2 fixed by going channel-0-only). RMSE/MAE/Dir Acc/Sharpe all got worse. This confirmed the flat/noisy loss curve isn't fixable by adding auxiliary targets naively.
3. **Patch length = 7 (current)** -- back to the channel-0-only loss (the only variant that hasn't shown a regression), testing a different lever from the `steven` branch: `patch_length=7`, `patch_stride=7` (non-overlapping), so each patch is exactly one trading day, matching `steven/src/models/patchtst.py`'s `PATCH_LEN=7` design -- instead of this notebook's previous formula-derived `patch_length=16`/`stride=8`.

**Question this variant tests**: does aligning patches to calendar days (rather than an arbitrary quarter-of-input_size length) change the loss curve's shape or `return_1h`'s test metrics?

**Caveat**: one seed, one trained model. `INPUT_SIZE=240` isn't an exact multiple of 7, so the patchifier will not use every bar in the context window evenly -- isolating the patch-length change alone rather than also re-tuning context length to be day-aligned.

In [1]:
# -- Colab Setup (skip automatically if running locally) --------------------------------------------
# Ongoing analysis tool, not a one-off experiment pinned to a historical commit
# (contrast with patchtst.ipynb / deepar.ipynb, which pin TARGET_COMMIT per
# docs/experiments.md's reproducibility workflow) -- always clones latest Model.
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'transformers', '-q'], check=True)

    REPO_URL = 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
    if not os.path.exists('/content/ECE1508_GenAI'):
        subprocess.run(
            ['git', 'clone', '--branch', 'Model', REPO_URL, '/content/ECE1508_GenAI'],
            check=True,
        )
    subprocess.run(['git', '-C', '/content/ECE1508_GenAI', 'fetch', 'origin'], check=True)
    # reset --hard + clean -fd (not checkout): the /content clone persists across
    # cell reruns in one Colab session, so a prior run's local changes would make
    # a plain `checkout`/`pull` fail or silently keep stale files.
    subprocess.run(['git', '-C', '/content/ECE1508_GenAI', 'reset', '--hard', 'origin/Model'], check=True)
    subprocess.run(['git', '-C', '/content/ECE1508_GenAI', 'clean', '-fd'], check=True)
    os.chdir('/content/ECE1508_GenAI/notebooks')

    missing = [
        f for f in ['train.parquet', 'val.parquet', 'test.parquet']
        if not os.path.exists(f'/content/ECE1508_GenAI/data/splits/{f}')
    ]
    if missing:
        raise FileNotFoundError('Missing data splits after clone: ' + ', '.join(missing))
    print('Data splits found from clone -- no upload needed.')

    import torch
    if torch.cuda.is_available():
        print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    else:
        print('\nNo GPU detected. Go to Runtime -> Change runtime type -> T4 GPU.')

print('Setup complete.')

Data splits found from clone -- no upload needed.

GPU: NVIDIA A100-SXM4-80GB
Setup complete.


In [2]:
import sys
sys.path.insert(0, '..')

import random
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PatchTSTConfig, PatchTSTForPrediction
from scipy.stats import t as student_t
import matplotlib.pyplot as plt

from scripts.models.metrics import compute_all

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

TARGET_COL = 'return_1h'
# Only the two groups patch_tst_feature_importance.ipynb found clearly, consistently
# helpful across both seeds: return_1h's own history (positive control) and OHLC price.
# Everything else (momentum, lagged returns, volatility/bollinger, volume, VIX, calendar)
# is dropped -- momentum and lagged returns were actively harmful, the rest were noise
# or only mildly positive.
ALL_COLS   = [TARGET_COL, 'open', 'high', 'low', 'close']   # 5 channels; return_1h is channel 0
INPUT_SIZE = 240   # same as the feature-importance notebook -- isolating the feature-set
                   # change, not re-litigating lookback length
MAX_STEPS  = 1000  # matches this project's "final training" budget elsewhere
plt.rcParams['figure.figsize'] = (14, 4)

# -- Seed control ------------------------------------------
# CHANGE THIS to rerun with a different seed. Controls model weight init and
# DataLoader shuffling (both read from torch's global RNG state).
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f'SEED = {SEED}')

Device: cuda
SEED = 0


In [ ]:
# -- Helpers (channel-0-only loss restored -- see markdown above for why the
# multi-channel-loss variant was reverted; only build_model's patch params changed) --

# One patch = one trading day (7 hourly bars), non-overlapping -- matches
# steven/src/models/patchtst.py's PATCH_LEN=7, instead of this notebook's previous
# patch_params() formula (patch_len=max(4,min(16,input_size//4)) -> 16 for
# INPUT_SIZE=240, stride=8, i.e. overlapping patches with no calendar meaning).
PATCH_LEN    = 7
PATCH_STRIDE = 7


class ZScoreScaler:
    """Fit on training data, transform any split with the same statistics."""
    def fit(self, data: np.ndarray):
        self.mean_ = data.mean(axis=0)
        self.std_  = data.std(axis=0) + 1e-8
        return self

    def transform(self, data: np.ndarray) -> np.ndarray:
        return (data - self.mean_) / self.std_

    def inverse_col0(self, z: np.ndarray) -> np.ndarray:
        return z * self.std_[0] + self.mean_[0]


class WindowDataset(Dataset):
    """
    past_values  : (context_length, num_channels)  -- lookback window, all (5) channels
    future_values: (1, num_channels)                -- all channels at t+1 (channel 0 is
                                                       the training target; see train_model)
    """
    def __init__(self, data: np.ndarray, context_length: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.ctx  = context_length

    def __len__(self):
        return len(self.data) - self.ctx

    def __getitem__(self, i):
        return {
            'past_values':   self.data[i : i + self.ctx],
            'future_values': self.data[i + self.ctx : i + self.ctx + 1],
        }


def build_model(input_size: int) -> PatchTSTForPrediction:
    config = PatchTSTConfig(
        num_input_channels=len(ALL_COLS),
        context_length=input_size,
        prediction_length=1,
        patch_length=PATCH_LEN,
        patch_stride=PATCH_STRIDE,   # real PatchTSTConfig param name (see docs/experiments.md --
                                     # `stride=` is silently absorbed as an unused kwarg by HF's
                                     # PretrainedConfig and falls back to the library default)
        d_model=128,
        num_attention_heads=8,
        num_hidden_layers=3,
        dropout=0.2,
        head_dropout=0.0,
        channel_attention=True,   # keeps the setup identical to the feature-importance
                                  # notebook's baseline; with only 5 correlated (OHLC+return)
                                  # channels left, this mostly lets price cross-inform return_1h
        loss='nll',
        distribution_output='student_t',
    )
    return PatchTSTForPrediction(config)


def _distribution_params(model: PatchTSTForPrediction, past_values: torch.Tensor):
    """
    Bypass model.head.forward()'s final tuple-transpose step, which assumes
    prediction_length > 1 and raises IndexError with a distribution head at
    prediction_length=1 (see docs/experiments.md / patch_tst_feature_importance.ipynb
    for the full verification against the transformers source).

    Uses mean pooling (pooling_type='mean', PatchTSTConfig's default) -- same as
    every other notebook this project compares against.
    """
    base_out = model.model(past_values=past_values)
    pooled   = base_out.last_hidden_state.mean(dim=2)             # mean-pool across patches: (bs, channels, d_model)
    pooled   = model.head.dropout(model.head.flatten(pooled))
    df, loc, scale = model.head.projection(pooled)                # ParameterProjection -> (bs, channels) each

    win_loc   = base_out.loc[:, 0, :]     # (bs, channels)
    win_scale = base_out.scale[:, 0, :]
    return df, loc, scale, win_loc, win_scale


def train_model(
    model: PatchTSTForPrediction,
    data: np.ndarray,
    context_length: int,
    max_steps: int,
    batch_size: int = 64,
    log_every: int = 10,
):
    """
    Same training loop as every other notebook in this project. Loss restricted
    to channel 0 (return_1h) only -- reverted back from the multi-channel-loss
    variant, which reintroduced a lag-echo pathology (see markdown above).

    Returns (model, loss_history) where loss_history is a list of (step, loss) pairs.
    """
    dataset = WindowDataset(data, context_length)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    opt     = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.to(DEVICE).train()
    loss_history = []
    step = 0
    while step < max_steps:
        for batch in loader:
            pv = batch['past_values'].to(DEVICE)
            fv = batch['future_values'].to(DEVICE)[:, 0, :]   # (bs, num_channels) -- squeeze prediction_length=1 axis

            df, loc, scale, win_loc, win_scale = _distribution_params(model, pv)

            # Loss restricted to channel 0 (return_1h) only, matching every other
            # notebook in this project (see docs/experiments.md Run 1/2).
            target0   = fv[:, 0]
            df0, loc0, scale0     = df[:, 0], loc[:, 0], scale[:, 0]
            win_loc0, win_scale0  = win_loc[:, 0], win_scale[:, 0]

            standardized_target = (target0 - win_loc0) / win_scale0
            base_dist = torch.distributions.StudentT(df=df0, loc=loc0, scale=scale0)
            log_prob  = base_dist.log_prob(standardized_target) - torch.log(win_scale0)
            loss = -log_prob.mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step += 1

            if step == 1 or step % log_every == 0:
                loss_history.append((step, float(loss.item())))
            if step >= max_steps:
                break
    return model, loss_history


def build_windows(full_data: np.ndarray, context_length: int, start_idx: int) -> np.ndarray:
    """(n, context_length, num_channels) windows, one per test-set prediction origin."""
    return np.stack([
        full_data[i - context_length : i]
        for i in range(start_idx, len(full_data))
    ])


def predict_from_windows(model: PatchTSTForPrediction, windows: np.ndarray, batch_size: int = 512):
    """
    Same analytic-forecast logic as predict_distribution below, but takes
    pre-built windows directly -- lets permutation-importance-style code feed in
    windows with specific channels shuffled, without rebuilding from scratch.
    """
    model.eval()
    means, lo_80, hi_80, lo_90, hi_90 = [], [], [], [], []
    loc_raw_list, win_loc_list, win_scale_list = [], [], []
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            x = torch.tensor(windows[i : i + batch_size], dtype=torch.float32).to(DEVICE)
            df, loc, scale, win_loc, win_scale = _distribution_params(model, x)

            df0, loc0, scale0 = df[:, 0].cpu().numpy(), loc[:, 0].cpu().numpy(), scale[:, 0].cpu().numpy()
            wloc0, wscale0    = win_loc[:, 0].cpu().numpy(), win_scale[:, 0].cpu().numpy()

            mean_z = loc0 * wscale0 + wloc0
            q = {lvl: student_t.ppf(lvl, df0, loc=loc0, scale=scale0) * wscale0 + wloc0
                 for lvl in [0.05, 0.10, 0.90, 0.95]}

            means.append(mean_z)
            lo_80.append(q[0.10]); hi_80.append(q[0.90])
            lo_90.append(q[0.05]); hi_90.append(q[0.95])
            loc_raw_list.append(loc0); win_loc_list.append(wloc0); win_scale_list.append(wscale0)

    return (np.concatenate(means), np.concatenate(lo_80), np.concatenate(hi_80),
            np.concatenate(lo_90), np.concatenate(hi_90),
            np.concatenate(loc_raw_list), np.concatenate(win_loc_list), np.concatenate(win_scale_list))


def predict_distribution(model, full_data, context_length, start_idx, batch_size=512):
    windows = build_windows(full_data, context_length, start_idx)
    return predict_from_windows(model, windows, batch_size)

In [4]:
# -- Load and normalise data (restricted to the reduced feature set) ---------------
train_raw = pd.read_parquet('../data/splits/train.parquet')
val_raw   = pd.read_parquet('../data/splits/val.parquet')
test_raw  = pd.read_parquet('../data/splits/test.parquet')

train_mat = train_raw[ALL_COLS].values.astype(np.float32)
val_mat   = val_raw[ALL_COLS].values.astype(np.float32)
test_mat  = test_raw[ALL_COLS].values.astype(np.float32)

scaler     = ZScoreScaler().fit(train_mat)
train_norm = scaler.transform(train_mat)
val_norm   = scaler.transform(val_mat)
test_norm  = scaler.transform(test_mat)

trainval_norm = np.concatenate([train_norm, val_norm], axis=0)
full_norm     = np.concatenate([train_norm, val_norm, test_norm], axis=0)

n_train, n_val, n_test = len(train_norm), len(val_norm), len(test_norm)

print(f'Train: {n_train:,}  Val: {n_val:,}  Test: {n_test:,}')
print(f'Input channels ({len(ALL_COLS)}): {ALL_COLS}')

Train: 21,028  Val: 1,744  Test: 2,469
Input channels (5): ['return_1h', 'open', 'high', 'low', 'close']


In [5]:
# -- Train the reduced-feature model --------------------------------
model, loss_history = train_model(
    build_model(INPUT_SIZE), trainval_norm, INPUT_SIZE, max_steps=MAX_STEPS,
)
print(f'Trained on {len(trainval_norm):,} rows, input_size={INPUT_SIZE}, '
      f'{len(loss_history)} logged loss points over {MAX_STEPS} steps')

Trained on 22,772 rows, input_size=240, 101 logged loss points over 1000 steps


In [ ]:
# -- Training loss curve ------------------------------------------
# Compare this curve's shape to the channel-0-only baseline's (patch_length=16,
# stride=8) flat/noisy pattern across the full 1000 steps. If day-aligned patches
# (patch_length=7, stride=7) change that shape, patch granularity/alignment is
# doing something to the optimization; if it looks the same, it isn't.
steps, losses = zip(*loss_history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses)
ax.set_xlabel('training step')
ax.set_ylabel('NLL loss (channel-0 only)')
ax.set_title('Training loss curve -- patch_length=7 (1 trading day), reduced feature set')
plt.tight_layout()
plt.show()

In [ ]:
# -- Test metrics (patch_length=7, reduced feature set) --------------------------------
test_start_idx = n_train + n_val
baseline_windows = build_windows(full_norm, INPUT_SIZE, test_start_idx)
z_test_true = test_norm[:, 0]

(y_pred_z, lo_80_z, hi_80_z, lo_90_z, hi_90_z,
 loc_raw, win_loc, win_scale) = predict_from_windows(model, baseline_windows)

y_true = scaler.inverse_col0(z_test_true)
y_pred = scaler.inverse_col0(y_pred_z)
lo_80  = scaler.inverse_col0(lo_80_z)
hi_80  = scaler.inverse_col0(hi_80_z)
lo_90  = scaler.inverse_col0(lo_90_z)
hi_90  = scaler.inverse_col0(hi_90_z)

results = compute_all(y_true, y_pred, lo_80, hi_80, lo_90, hi_90)

print('=== patch_length=7 (1 trading day), reduced feature set (return_1h + OHLC, 5 channels) ===')
print(f"  RMSE              : {results['rmse']:.6f}")
print(f"  MAE               : {results['mae']:.6f}")
print(f"  Directional Acc   : {results['dir_acc']:.4f}")
print(f"  Coverage 80%      : {results['coverage_80']:.4f}  (target: 0.80)")
print(f"  Coverage 90%      : {results['coverage_90']:.4f}  (target: 0.90)")
print(f"  Sharpe Ratio      : {results['sharpe']:.4f}")
print(f"  Max Drawdown      : {results['max_drawdown']:.6f}")

y_lag1_norm  = np.roll(z_test_true, 1)
corr_loc_raw = np.corrcoef(loc_raw[1:], y_lag1_norm[1:])[0, 1]
print(f"\n  corr(loc_raw, y[t-1]) : {corr_loc_raw:.4f}   <- lag-echo diagnostic, for reference against docs/experiments.md")

print('\n=== For comparison: channel-0-only baseline, patch_length=16/stride=8 (SEED=0, commit eaaa35e) ===')
print('  RMSE 0.004148 / MAE 0.002342 / Dir Acc 0.5318 / Sharpe 0.8098')
print('=== For comparison: multi-channel NLL loss variant, patch_length=16/stride=8 (SEED=0) ===')
print('  RMSE 0.004332 / MAE 0.002523 / Dir Acc 0.4998 / Sharpe -0.4828  (reverted -- lag-echo regression)')
print('=== For comparison: patch_tst_feature_importance.ipynb 21-channel baseline (SEED=0, commit de8f191) ===')
print('  RMSE 0.004147 / MAE 0.002342 / Dir Acc 0.5269 / Sharpe 0.8229')

## Caveats and recommended next steps

- **One seed, one trained model** -- rerun with a different `SEED` before trusting any specific delta against the channel-0-only baseline (`eaaa35e`) or the 21-channel baseline (`de8f191`).
- **This isolates one variable at a time on purpose.** Patch length/stride is the only thing changed vs. the channel-0-only baseline -- loss scope, distribution head, pooling, and feature set are all unchanged, so any delta here is attributable to patch granularity/alignment specifically.
- **`INPUT_SIZE=240` is not a multiple of 7**, so day-aligned, non-overlapping patches won't cleanly tile the whole context window -- if this looks promising, the natural follow-up is also snapping `INPUT_SIZE` to a multiple of 7 (e.g. 238 = 34 days) rather than treating patch length in isolation forever.
- **If this doesn't clearly beat the channel-0-only baseline**, remember patch alignment doesn't add new information either -- feature-ablation work already showed `return_1h` + OHLC sits at this project's information ceiling. The most this experiment can show is whether patch granularity affects the *optimization*, not whether there's more signal to find.
- **No transaction costs are modeled** in this project's Sharpe/Max Drawdown calculation (`scripts/models/metrics.py`) -- keep that in mind if any delta here is framed in terms of trading performance.